<a href="https://colab.research.google.com/github/khushi-2003/AI-projects/blob/main/TextGenerationModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Text Generation Model
1. IMPORT LIBRARIES

In [ ]:
! pip install tensorflow

In [ ]:
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize


from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, LSTM, Embedding, Dropout
from tensorflow.keras.callbacks import EarlyStopping

Data Gathering

In [ ]:
with open("/content/balanced_ai_human_prompts.csv", "r", encoding="utf-8") as file:
    text = file.read()

print(text[:500])

text,generated
"Machine learning, a subset of artificial intelligence, has rapidly emerged as a transformative force, revolutionizing industries and redefining the possibilities of technology. At its core, machine learning enables computers to learn from data and make informed decisions without explicit programming, with applications ranging from image recognition and language processing to autonomous systems. As machine learning continues to advance, it brings both opportunities and challenges,


In [ ]:
print(len(text))

4616849


Tokenization

In [ ]:
text = text[:20000]
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1
print("Vocabulary Size:", total_words)

# Reverse mapping: index -> word
index_word = {i: word for word, i in tokenizer.word_index.items()}


Vocabulary Size: 1136


In [ ]:
tokenizer.word_index

{'the': 1,
 'and': 2,
 'of': 3,
 'a': 4,
 'to': 5,
 'in': 6,
 'as': 7,
 'from': 8,
 '1': 9,
 'with': 10,
 'for': 11,
 'chess': 12,
 'have': 13,
 'its': 14,
 'world': 15,
 'has': 16,
 'learning': 17,
 'data': 18,
 'technology': 19,
 'advancements': 20,
 'science': 21,
 'by': 22,
 'machine': 23,
 'is': 24,
 'decision': 25,
 'systems': 26,
 'challenges': 27,
 'making': 28,
 'such': 29,
 'that': 30,
 'human': 31,
 'including': 32,
 'understanding': 33,
 'become': 34,
 'into': 35,
 'intelligence': 36,
 'applications': 37,
 'it': 38,
 'both': 39,
 'shaping': 40,
 'digital': 41,
 'becomes': 42,
 'devices': 43,
 'on': 44,
 'our': 45,
 'complex': 46,
 'calculus': 47,
 'physics': 48,
 'energy': 49,
 'role': 50,
 'global': 51,
 'matter': 52,
 'chemistry': 53,
 'artificial': 54,
 'computers': 55,
 'interconnected': 56,
 'or': 57,
 'education': 58,
 'societal': 59,
 'beyond': 60,
 'knowledge': 61,
 'modern': 62,
 'computing': 63,
 'quantum': 64,
 'ethical': 65,
 'evolution': 66,
 'testament': 67,
 

CREATE INPUT SEQUENCES : Instead of splitting line by line, use continuous text for better learning

In [ ]:
text[:50]

'text,generated\n"Machine learning, a subset of arti'

In [ ]:
token_list = tokenizer.texts_to_sequences([text])[0]

input_sequences = []

for i in range(4, len(token_list)):
    n_gram_sequence = token_list[i-4:i+1]
    input_sequences.append(n_gram_sequence)

print(input_sequences[:5])


[[383, 384, 23, 17, 4], [384, 23, 17, 4, 385], [23, 17, 4, 385, 3], [17, 4, 385, 3, 54], [4, 385, 3, 54, 36]]


In [ ]:
token_list

[383,
 384,
 23,
 17,
 4,
 385,
 3,
 54,
 36,
 16,
 386,
 199,
 7,
 4,
 200,
 201,
 387,
 79,
 2,
 388,
 1,
 126,
 3,
 19,
 80,
 14,
 389,
 23,
 17,
 390,
 55,
 5,
 202,
 8,
 18,
 2,
 391,
 203,
 127,
 392,
 393,
 394,
 10,
 37,
 204,
 8,
 395,
 205,
 2,
 206,
 81,
 5,
 207,
 26,
 7,
 23,
 17,
 82,
 5,
 396,
 38,
 397,
 39,
 208,
 2,
 27,
 10,
 398,
 3,
 209,
 210,
 2,
 128,
 211,
 399,
 400,
 1,
 212,
 3,
 23,
 17,
 24,
 129,
 40,
 4,
 130,
 401,
 213,
 26,
 214,
 5,
 131,
 83,
 2,
 4,
 84,
 56,
 15,
 9,
 4,
 25,
 215,
 4,
 132,
 23,
 17,
 216,
 402,
 25,
 28,
 6,
 4,
 215,
 85,
 217,
 218,
 403,
 127,
 404,
 405,
 406,
 2,
 407,
 408,
 219,
 127,
 57,
 409,
 1,
 216,
 410,
 411,
 412,
 80,
 220,
 413,
 221,
 414,
 1,
 415,
 5,
 222,
 218,
 416,
 4,
 417,
 221,
 24,
 418,
 419,
 1,
 219,
 25,
 223,
 224,
 420,
 39,
 421,
 2,
 422,
 18,
 86,
 423,
 87,
 4,
 225,
 424,
 226,
 25,
 227,
 425,
 426,
 8,
 427,
 428,
 87,
 228,
 85,
 429,
 2,
 430,
 431,
 229,
 27,
 25,
 227,
 133,
 7,
 432

PADDING

SPLIT X AND Y

In [ ]:
max_sequence_length = 6

input_sequences = np.array(input_sequences)

X = input_sequences[:, :-1]
y = input_sequences[:, -1]


# Converting target to one-hot encoding
y = to_categorical(y, num_classes=total_words)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (2727, 4)
y shape: (2727, 1136)


BUILD LSTM MODEL

In [ ]:
model = Sequential()

# Correct input shape = sequence length - 1
model.add(Input(shape=(max_sequence_length - 1,))) # 5

# Embedding layer
model.add(Embedding(input_dim=total_words, output_dim=128))

# First LSTM layer
model.add(LSTM(150, return_sequences=True, dropout=0.2))

# Second LSTM layer
model.add(LSTM(100, dropout=0.2))

# Hidden Dense layer
model.add(Dense(100, activation='relu'))

# Output layer
model.add(Dense(total_words, activation='softmax')) #units=6032

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 5, 128)         │       145,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_6 (LSTM)                   │ (None, 5, 150)         │       167,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_7 (LSTM)                   │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1136)           │       114,736 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 538,044 (2.05 MB)

 Trainable params: 538,044 (2.05 MB)

 Non-trainable params: 0 (0.00 B)

Model Training

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='loss',patience=3,restore_best_weights=True)

history = model.fit(X,y,epochs=30,batch_size=32,verbose=1,callbacks=[early_stop])


Epoch 1/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.0521 - loss: 6.7308
Epoch 2/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.0539 - loss: 6.2202
Epoch 3/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.0568 - loss: 6.1137
Epoch 4/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.0539 - loss: 6.0213
Epoch 5/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.0535 - loss: 5.9516
Epoch 6/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.0532 - loss: 5.8659
Epoch 7/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.0645 - loss: 5.7520
Epoch 8/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.0653 - loss: 5.6201
Epoch 9/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.0755 - loss: 5.5049
Epoch 10/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.0865 - loss: 5.4144
Epoch 11/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.0902 - loss: 5.2881
Epoch 12/30
86/86 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.1041 - lo

Using the trained model for final predictions
TEMPERATURE + TOP-K SAMPLING

In [ ]:
model.save("TextGenerationModel.keras")

In [ ]:
def sample_with_temperature(preds, temperature=0.8, top_k=4):
    preds = np.asarray(preds).astype("float64")

    # Selecting top k probabilities
    top_indices = np.argsort(preds)[-top_k:]
    top_probs = preds[top_indices]

    # Applying temp scaling
    top_probs = np.log(top_probs + 1e-10) / temperature
    exp_probs = np.exp(top_probs)
    top_probs = exp_probs / np.sum(exp_probs)

    return np.random.choice(top_indices, p=top_probs)

Text Generation method

In [ ]:
def generate_text(seed_text, next_words=20):
    output_text = seed_text
    generated_words = []

    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([output_text])[0]

        token_list = pad_sequences(
            [token_list],
            maxlen=max_sequence_length - 1,
            padding='pre'
        )

        predicted_probs = model.predict(token_list, verbose=0)[0]

        predicted_index = sample_with_temperature(
            predicted_probs,
            temperature=0.9,
            top_k=4
        )

        next_word = index_word.get(predicted_index, "")

        if next_word in generated_words[-2:]:
            continue

        generated_words.append(next_word)
        output_text += " " + next_word

    return output_text

Predictions

In [ ]:
print(generate_text("Machine Learning is ", next_words=15))

Machine Learning is  integral to education the advent remains artificial intelligence technology tasks audiences and statistical becomes
